In [21]:
import glob
import os
import re
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import yt

# Point to your GRChombo plotfile directory via params
# (reads output_path and hdf5_subpath from the params file)
root_dir = Path("../GRChombo-tests/Examples/DW_flat/")
params_file = root_dir/ "GRTresna_restart_setup/params_grchombo_128.txt"

def _read_param_value(lines, key):
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.split()[0] == key:
            _, val = line.split("=", 1)
            val = val.split("#", 1)[0]
            return val.strip().strip('"')
    return None

param_lines = params_file.read_text().splitlines()
output_path = _read_param_value(param_lines, "output_path")
hdf5_subpath = _read_param_value(param_lines, "hdf5_subpath")
if output_path is None or hdf5_subpath is None:
    raise ValueError("Missing output_path or hdf5_subpath in params file.")

plot_dir = str(root_dir / output_path / hdf5_subpath)

# Collect plotfiles
plot_files = sorted(glob.glob(os.path.join(plot_dir, "Cosmop_*.3d.hdf5")))
if not plot_files:
    raise FileNotFoundError(f"No Cosmop_*.3d.hdf5 files found in {plot_dir}")

print(f"plot_dir: {plot_dir}")
print(f"plot_files: {len(plot_files)}")

plot_files[:3], plot_files[-1]

plot_dir: ../GRChombo-tests/Examples/DW_flat/Output_GRChombo_128/hdf5_GRChombo_DW_flat_128
plot_files: 6


(['../GRChombo-tests/Examples/DW_flat/Output_GRChombo_128/hdf5_GRChombo_DW_flat_128/Cosmop_000000.3d.hdf5',
  '../GRChombo-tests/Examples/DW_flat/Output_GRChombo_128/hdf5_GRChombo_DW_flat_128/Cosmop_000001.3d.hdf5',
  '../GRChombo-tests/Examples/DW_flat/Output_GRChombo_128/hdf5_GRChombo_DW_flat_128/Cosmop_000002.3d.hdf5'],
 '../GRChombo-tests/Examples/DW_flat/Output_GRChombo_128/hdf5_GRChombo_DW_flat_128/Cosmop_000005.3d.hdf5')

In [22]:
# List fields present in the plotfile

def list_fields(file_name):
    fields = []
    with h5py.File(file_name, "r") as f:
        for key, value in f.attrs.items():
            if re.search(r"^component", key):
                name = value.decode("utf-8") if isinstance(value, bytes) else str(value)
                fields.append(name)
    return fields

fields = list_fields(plot_files[-1])
fields

['chi',
 'phi',
 'rho_scaled',
 'S_scaled',
 'K_scaled',
 'lapse',
 'Ham',
 'Mom',
 'Ham_abs',
 'Mom_abs']

In [23]:
# Movie export settings

output_dir = Path("./movies")
output_dir.mkdir(parents=True, exist_ok=True)

# Sampling resolution for plots
n = 128

# Skip frames to speed up movie creation
step_stride = 1  # use 2 or 4 to skip frames

# Choose format: "gif"
movie_format = "gif"

# Use a consistent color scale across time
global_scale = True

# Overlay grid boundaries on movie frames (only for native plotfile fields)
annotate_grids = False

In [24]:
# Helper to extract a midplane slice for a given field

def extract_slice(file_name, field_name, n=128):
    ds = yt.load(file_name)
    dd = ds.r[::n*1j, ::n*1j, ::n*1j]
    data = dd[field_name]
    z_index = data.shape[2] // 2
    return data[:, :, z_index]

In [25]:
# Resolve field names for yt (handles tuple-style names)
def _resolve_field(ds, field_name):
    if field_name in ds.field_list:
        return field_name
    for f in ds.field_list:
        if isinstance(f, tuple) and len(f) > 1 and f[1] == field_name:
            return f
    return field_name

In [26]:
# Helper to compute normalized constraints: Ham/Ham_abs and Mom/Mom_abs

def extract_constraint_ratios(file_name, n=128, eps=1e-30):
    ham = extract_slice(file_name, "Ham", n=n)
    ham_abs = extract_slice(file_name, "Ham_abs", n=n)
    mom = extract_slice(file_name, "Mom", n=n)
    mom_abs = extract_slice(file_name, "Mom_abs", n=n)

    ham_norm = ham / (np.abs(ham_abs) + eps)
    mom_norm = mom / (np.abs(mom_abs) + eps)
    return ham_norm, mom_norm

In [27]:
# Compute global vmin/vmax per field (optional but recommended)

movie_fields = list(fields)
use_norm_constraints = "Ham_abs" in fields and "Mom_abs" in fields
if use_norm_constraints:
    # movie_fields = [f for f in movie_fields if f not in ("Ham", "Mom")]
    movie_fields.extend(["Ham_rel", "Mom_rel"])
    print("Using normalized constraints for movies: Ham_rel, Mom_rel")
else:
    print("Ham_abs/Mom_abs not found; using raw Ham/Mom in movies")

vminmax = {}
if global_scale:
    for field in movie_fields:
        mins = []
        maxs = []
        for fname in plot_files[::step_stride]:
            if field == "Ham_rel":
                arr, _ = extract_constraint_ratios(fname, n=n)
            elif field == "Mom_rel":
                _, arr = extract_constraint_ratios(fname, n=n)
            else:
                arr = extract_slice(fname, field, n=n)
            mins.append(np.nanmin(arr))
            maxs.append(np.nanmax(arr))
        vminmax[field] = (min(mins), max(maxs))
else:
    vminmax = {field: (None, None) for field in movie_fields}

vminmax

yt : [WARNING  ] 2026-03-09 20:58:26,472 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 20:58:26,473 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 20:58:26,474 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 20:58:26,535 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-09 20:58:26,536 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 20:58:26,537 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 20:58:26,538 Parameters: domain_right_edge         = [4. 4. 4.]


Using normalized constraints for movies: Ham_rel, Mom_rel


yt : [WARNING  ] 2026-03-09 20:58:28,343 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 20:58:28,343 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 20:58:28,344 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 20:58:28,392 Parameters: current_time              = 0.0078125
yt : [INFO     ] 2026-03-09 20:58:28,393 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 20:58:28,393 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 20:58:28,394 Parameters: domain_right_edge         = [4. 4. 4.]
yt : [WARNING  ] 2026-03-09 20:58:29,955 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 20:58:29,956 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 20:58:29,957 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 20:58:30,004 Parameters: current_time              = 0.015625
yt : [INFO     ] 2026-03-09 20:58:30,005 Parameters: domain_dimensions        

{'chi': (unyt_quantity(0.83134739, '(dimensionless)'),
  unyt_quantity(0.99999334, '(dimensionless)')),
 'phi': (unyt_quantity(0.00271651, '(dimensionless)'),
  unyt_quantity(0.04318677, '(dimensionless)')),
 'rho_scaled': (unyt_quantity(1.07025616e-28, '(dimensionless)'),
  unyt_quantity(0.03575992, '(dimensionless)')),
 'S_scaled': (unyt_quantity(-0.0748174, '(dimensionless)'),
  unyt_quantity(-2.02145616e-28, '(dimensionless)')),
 'K_scaled': (unyt_quantity(-9.51969211e-05, '(dimensionless)'),
  unyt_quantity(1.88173504, '(dimensionless)')),
 'lapse': (unyt_quantity(0.89469009, '(dimensionless)'),
  unyt_quantity(1., '(dimensionless)')),
 'Ham': (unyt_quantity(-0.06121763, '(dimensionless)'),
  unyt_quantity(0.01928871, '(dimensionless)')),
 'Mom': (unyt_quantity(3.27151609e-08, '(dimensionless)'),
  unyt_quantity(0.05388487, '(dimensionless)')),
 'Ham_abs': (unyt_quantity(0.00025212, '(dimensionless)'),
  unyt_quantity(3.92427514, '(dimensionless)')),
 'Mom_abs': (unyt_quantity(1.1

In [28]:
# Compute interior L2 norms of Ham_rel/Mom_rel over time and save to txt

def _l2_norm_masked(arr, mask):
    vals = arr[mask]
    return float(np.sqrt(np.mean(vals * vals)))

def _central_region_mask_2d(n):
    lo = int(np.floor(n * 1.0 / 8.0))
    hi = int(np.ceil(n * 7.0 / 8.0))
    mask = np.zeros((n, n), dtype=bool)
    mask[lo:hi, lo:hi] = True
    return mask

def _read_time_from_hdf5(file_name):
    with h5py.File(file_name, "r") as f:
        if "time" in f.attrs:
            return float(f.attrs["time"])
    return np.nan

if "Ham_abs" in fields and "Mom_abs" in fields:
    times = []
    ham_l2 = []
    mom_l2 = []
    mask = _central_region_mask_2d(n)

    for fname in plot_files[::step_stride]:
        t = _read_time_from_hdf5(fname)
        ham_norm, mom_norm = extract_constraint_ratios(fname, n=n)
        times.append(t)
        ham_l2.append(_l2_norm_masked(ham_norm, mask))
        mom_l2.append(_l2_norm_masked(mom_norm, mask))

    out_path = output_dir / "constraint_rel_l2.txt"
    data = np.column_stack([times, ham_l2, mom_l2])
    np.savetxt(
        out_path,
        data,
        header="time Ham_rel_L2 Mom_rel_L2",
    )
    print(f"Wrote {out_path}")
else:
    print("Ham_abs/Mom_abs not found; skipping relative L2 output.")

yt : [WARNING  ] 2026-03-09 21:00:02,412 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:00:02,413 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:00:02,413 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:00:02,453 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-09 21:00:02,454 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 21:00:02,455 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 21:00:02,456 Parameters: domain_right_edge         = [4. 4. 4.]
yt : [WARNING  ] 2026-03-09 21:00:03,583 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:00:03,584 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:00:03,585 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:00:03,619 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-09 21:00:03,620 Parameters: domain_dimensions         = [128 128

Wrote movies/constraint_rel_l2.txt


In [ ]:
# Build a movie for each field (stream frames to avoid high memory use)

import imageio.v2 as imageio

if movie_format != "gif":
    raise ValueError("movie_format must be 'gif'")

for field in movie_fields:
    out_path = output_dir / f"{field}.gif"

    with imageio.get_writer(out_path, mode="I", fps=4, loop=0) as writer:
        for fname in plot_files[::step_stride]:
            vmin, vmax = vminmax[field]

            if field in ("Ham_rel", "Mom_rel"):
                if field == "Ham_rel":
                    arr, _ = extract_constraint_ratios(fname, n=n)
                else:
                    _, arr = extract_constraint_ratios(fname, n=n)

                fig, ax = plt.subplots(figsize=(5, 4), dpi=120)
                im = ax.imshow(arr, origin="lower", vmin=vmin, vmax=vmax)
                ax.set_title(f"{field} | {os.path.basename(fname)}")
                fig.colorbar(im, ax=ax)
                fig.canvas.draw()
                image = np.asarray(fig.canvas.buffer_rgba())[..., :3]
                writer.append_data(image)
                plt.close(fig)
                continue

            if annotate_grids:
                ds = yt.load(fname)
                field_yt = _resolve_field(ds, field)
                p = yt.SlicePlot(ds, "z", field_yt)
                p.set_zlim(field_yt, vmin, vmax)
                p.annotate_grids()
                p.set_axes_unit("code_length")
                p.set_colorbar_label(field_yt, field)
                p.annotate_title(f"{field} | {os.path.basename(fname)}")
                tmp_path = output_dir / "_frame_tmp.png"
                p.save(str(tmp_path))
                image = imageio.imread(tmp_path)
                writer.append_data(image)
            else:
                arr = extract_slice(fname, field, n=n)
                fig, ax = plt.subplots(figsize=(5, 4), dpi=120)
                im = ax.imshow(arr, origin="lower", vmin=vmin, vmax=vmax)
                ax.set_title(f"{field} | {os.path.basename(fname)}")
                fig.colorbar(im, ax=ax)
                fig.canvas.draw()
                image = np.asarray(fig.canvas.buffer_rgba())[..., :3]
                writer.append_data(image)
                plt.close(fig)

    print(f"Wrote {out_path}")

yt : [WARNING  ] 2026-03-09 21:00:21,412 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:00:21,413 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:00:21,413 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:00:21,447 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-09 21:00:21,448 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 21:00:21,449 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 21:00:21,450 Parameters: domain_right_edge         = [4. 4. 4.]
yt : [WARNING  ] 2026-03-09 21:00:22,249 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:00:22,250 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:00:22,251 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:00:22,286 Parameters: current_time              = 0.0078125
yt : [INFO     ] 2026-03-09 21:00:22,287 Parameters: domain_dimensions         = [1

Wrote movies/chi.gif


yt : [WARNING  ] 2026-03-09 21:00:27,832 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:00:27,833 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:00:27,834 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:00:27,867 Parameters: current_time              = 0.0078125
yt : [INFO     ] 2026-03-09 21:00:27,868 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 21:00:27,869 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 21:00:27,869 Parameters: domain_right_edge         = [4. 4. 4.]
yt : [WARNING  ] 2026-03-09 21:00:28,932 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:00:28,933 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:00:28,933 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:00:28,967 Parameters: current_time              = 0.015625
yt : [INFO     ] 2026-03-09 21:00:28,968 Parameters: domain_dimensions        

Wrote movies/phi.gif


yt : [WARNING  ] 2026-03-09 21:00:33,440 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:00:33,441 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:00:33,442 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:00:33,477 Parameters: current_time              = 0.0078125
yt : [INFO     ] 2026-03-09 21:00:33,477 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 21:00:33,478 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 21:00:33,479 Parameters: domain_right_edge         = [4. 4. 4.]
yt : [WARNING  ] 2026-03-09 21:00:34,275 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:00:34,276 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:00:34,277 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:00:34,310 Parameters: current_time              = 0.015625
yt : [INFO     ] 2026-03-09 21:00:34,311 Parameters: domain_dimensions        

...

yt : [WARNING  ] 2026-03-09 21:01:12,721 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:01:12,722 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:01:12,722 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:01:12,758 Parameters: current_time              = 0.0078125
yt : [INFO     ] 2026-03-09 21:01:12,759 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 21:01:12,759 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 21:01:12,760 Parameters: domain_right_edge         = [4. 4. 4.]
yt : [WARNING  ] 2026-03-09 21:01:13,557 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:01:13,558 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:01:13,559 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:01:13,593 Parameters: current_time              = 0.015625
yt : [INFO     ] 2026-03-09 21:01:13,594 Parameters: domain_dimensions        

Wrote movies/Mom_abs.gif


yt : [WARNING  ] 2026-03-09 21:01:17,865 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:01:17,865 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:01:17,866 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:01:17,902 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-09 21:01:17,902 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 21:01:17,904 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 21:01:17,905 Parameters: domain_right_edge         = [4. 4. 4.]
yt : [WARNING  ] 2026-03-09 21:01:18,989 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:01:18,990 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:01:18,990 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:01:19,025 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-09 21:01:19,025 Parameters: domain_dimensions         = [128 128

Wrote movies/Ham_rel.gif


yt : [WARNING  ] 2026-03-09 21:01:37,999 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:01:38,000 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:01:38,001 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:01:38,034 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-09 21:01:38,035 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-09 21:01:38,036 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-09 21:01:38,037 Parameters: domain_right_edge         = [4. 4. 4.]
yt : [WARNING  ] 2026-03-09 21:01:38,743 Setting code length unit to be 1.0 cm
yt : [WARNING  ] 2026-03-09 21:01:38,744 Setting code mass unit to be 1.0 g
yt : [WARNING  ] 2026-03-09 21:01:38,745 Setting code time unit to be 1.0 s
yt : [INFO     ] 2026-03-09 21:01:38,778 Parameters: current_time              = 0.0
yt : [INFO     ] 2026-03-09 21:01:38,779 Parameters: domain_dimensions         = [128 128

In [ ]:
# Display all movies in the output folder (GIFs and MP4s)
from IPython.display import HTML, display

movie_files = sorted(output_dir.glob("*"))
html_lines = []
for path in movie_files:
    suffix = path.suffix.lower()
    if suffix == ".gif":
        html_lines.append(f"<div><img src='{path.as_posix()}' /></div>")
    elif suffix == ".mp4":
        html_lines.append(
            f"<div><video src='{path.as_posix()}' controls></video></div>"
        )

display(HTML("\n".join(html_lines)))